# 🎬 TeraBox / 1024terabox Video Downloader & Streamer

**⚠️ Disclaimer:** Scrapping iteraplay.com is only for personal use.
**Made by:** Dr. Dev || Dr. Hamza
**Contact:** [t.me/Hamza3895](https://t.me/Hamza3895)
**Copyright © Dr. Dev || Dr. Hamza 2026. All rights reserved.**

---

### 📖 How to Use This Notebook

1. **Get your Cookies:** Export your `cookies.txt` using a browser extension on your PC while logged into TeraBox.
2. **Upload to Colab:** Click the Folder icon 📁 on the left sidebar and upload `cookies.txt` into the `/content/` area.
3. **Run Cell 2 (Dependencies):** Click the play button to install the required background packages. You only need to do this once.
4. **Run Cell 3 (Main Tool):** Click the play button to start the script and follow the text prompts to watch or download your video.

In [3]:

# ============================================================
# INSTALL DEPENDENCIES
# Run this cell once before using the main tool.
# ============================================================
!pip install curl_cffi tqdm -q
!apt-get install -y ffmpeg -q
print("✅ Dependencies installed successfully!")

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✅ Dependencies installed successfully!


In [4]:

!pip install curl_cffi tqdm -q
!apt-get install -y ffmpeg -q

# ============================================================
# Cell 2: Core functions
# ============================================================
import os, re, time, subprocess, shutil
from tqdm import tqdm
from curl_cffi import requests as cf_requests
from IPython.display import display, HTML, Video

SESSION = cf_requests.Session(impersonate="chrome124")

HEADERS = {
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "en-US,en;q=0.9",
    "user-agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
}

API_HEADERS = {
    "accept": "*/*",
    "accept-language": "en-US,en;q=0.9",
    "content-type": "application/json",
    "origin": "https://iteraplay.com",
    "referer": "https://iteraplay.com/",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
    "user-agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
}

def load_netscape_cookies(cookie_file: str) -> dict:
    """Parse Netscape cookie file → dict of {name: value}."""
    cookies = {}
    with open(cookie_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) >= 7:
                cookies[parts[5]] = parts[6]
    return cookies

def warm_up_session(cookie_file: str = None):
    print("🌐 Step 1: Fresh homepage visit (getting new session_id for this IP)...")
    r = SESSION.get("https://iteraplay.com/", headers=HEADERS, timeout=30)
    print(f"   Status: {r.status_code}")
    new_cookies = dict(SESSION.cookies)
    print(f"   New session cookies: {list(new_cookies.keys())}")
    time.sleep(1)

    if cookie_file and os.path.exists(cookie_file):
        all_cookies = load_netscape_cookies(cookie_file)

        # Inject ONLY login_token + remember_me
        inject = {k: v for k, v in all_cookies.items()
                  if k in ("login_token", "remember_me")}
        for name, value in inject.items():
            SESSION.cookies.set(name, value, domain="iteraplay.com")
        print(f"🍪 Step 2: Injected login cookies: {list(inject.keys())}")

        print("🔄 Step 3: Re-visiting to bind account to new session...")
        r2 = SESSION.get("https://iteraplay.com/", headers=HEADERS, timeout=30)
        print(f"   Status: {r2.status_code}")
        final_cookies = dict(SESSION.cookies)
        print(f"   Final cookies: {list(final_cookies.keys())}")
        time.sleep(1)
    else:
        print("   ⚠️  No cookie file — running as guest (5 req / 6h limit)")

def get_video_info(url: str, retries: int = 3, wait: int = 5) -> dict:
    for attempt in range(1, retries + 1):
        resp = SESSION.post(
            "https://iteraplay.com/api/download",
            headers=API_HEADERS,
            json={"url": url},
            timeout=30
        )
        print(f"   API status: {resp.status_code}  (attempt {attempt}/{retries})")

        if resp.status_code == 429:
            try:
                body = resp.json()
                usage = body.get("usage", {})
                print(f"   ⚠️  Rate limit — {usage.get('current')}/{usage.get('limit')} used, "
                      f"resets in ~{usage.get('resetHours')}h")
            except Exception:
                print("   ⚠️  Rate limited (429)")
            if attempt < retries:
                print(f"   ⏳ Waiting {wait}s...")
                time.sleep(wait)
                wait = min(wait * 2, 120)
                continue
            else:
                resp.raise_for_status()

        resp.raise_for_status()
        data = resp.json()

        if data.get("status") != "success":
            print("   Raw response:", data)
            raise ValueError(f"API error: {data.get('status')}")

        files = data.get("list", [])
        if not files:
            raise ValueError("No files returned")

        file_info = files[0]
        name = file_info.get("name", "")

        if any(kw in name for kw in ("Token", "token", "expired", "mismatch", "refresh", "Cookie")):
            print(f"   ⚠️  Server says: {name}")
            if attempt < retries:
                print("   🔄 Re-initialising session and retrying...")
                SESSION.cookies.clear()
                warm_up_session(cookie_file=COOKIE_FILE)
                time.sleep(3)
                continue
            else:
                raise RuntimeError(
                    f"Session error after {retries} attempts: {name}\n"
                    "💡 Your cookies have likely expired. Please export fresh cookies "
                    "from your browser and re-upload cookies.txt."
                )

        usage = data.get("usage", {})
        if usage:
            utype = usage.get("userType", "N/A")
            print(f"   📊 Usage: {usage.get('current')}/{usage.get('limit')} "
                  f"(resets ~{usage.get('resetHours')}h) — type: {utype}")
            if utype == "guest":
                print("   ⚠️  Still logged in as guest — login_token may be expired")

        return file_info

    raise RuntimeError("Failed after all retries.")

def show_info(info: dict):
    print(f"\n📄 Name     : {info.get('name', 'N/A')}")
    print(f"📦 Size     : {info.get('size_formatted', 'N/A')}")
    print(f"🎞  Quality  : {info.get('quality', 'N/A')}")
    print(f"⏱  Duration : {info.get('duration', 'N/A')}")
    streams = info.get("fast_stream_url", {})
    if isinstance(streams, dict) and streams:
        print(f"📡 Streams  : {', '.join(streams.keys())}")
    else:
        print(f"📡 Streams  : None")

# ============================================================
# Cell 3: Init → Paste URL → Watch or Download
# ============================================================

COOKIE_FILE = "/content/cookies.txt"   # ← your uploaded cookie file

warm_up_session(cookie_file=COOKIE_FILE)

url = input("\n🔗 Paste your TeraBox / 1024terabox URL: ").strip()

print("\nFetching video info...")
info = get_video_info(url)
show_info(info)

print("\nWhat do you want to do?")
print("  1 → Watch  (show url to play video)")
print("  2 → Download  (download via m3u8 stream to Colab & Drive)")
choice = input("Enter 1 or 2: ").strip()

# ============================================================
# Cell 4a: WATCH
# ============================================================
if choice == "1":
    streams = info.get("fast_stream_url", {})
    if not isinstance(streams, dict) or not streams:
        print("❌ No stream URLs available.")
    else:
        qualities = sorted(streams.keys(), key=lambda q: int(q.replace("p", "")))
        print(f"\nAvailable qualities: {', '.join(qualities)}")
        quality = input(f"Pick quality (default = {qualities[-1]}): ").strip()
        if quality not in streams:
            quality = qualities[-1]
            print(f"→ Using {quality}")

        m3u8_url = streams[quality]
        print(f"\n▶ Stream URL ({quality}):\n{m3u8_url}")
        print("\n🔗 Open this link in your browser or a network stream player (like VLC) to play the video.")

# ============================================================
# Cell 4b: DOWNLOAD via m3u8 (with Progress Bar & Google Drive)
# ============================================================
elif choice == "2":
    # 1. Connect to Google Drive first
    from google.colab import drive
    print("\nConnecting to Google Drive...")
    drive.mount('/content/drive')

    # 2. Ask for Quality
    streams = info.get("fast_stream_url", {})
    if not isinstance(streams, dict) or not streams:
        print("❌ No stream URLs found.")
    else:
        qualities = sorted(streams.keys(), key=lambda q: int(q.replace("p", "")))
        print(f"\nAvailable qualities: {', '.join(qualities)}")
        quality = input(f"Pick quality (default = {qualities[-1]}): ").strip()
        if quality not in streams:
            quality = qualities[-1]
            print(f"→ Using {quality}")

        m3u8_url = streams[quality]
        print(f"\n📡 m3u8 URL ({quality}):\n   {m3u8_url}")

        filename = re.sub(r'[\\/*?:"<>|]', "_", info.get("name", "video"))
        filename = re.sub(r'\.mp4$', '', filename, flags=re.IGNORECASE)

        out_path_colab = f"/content/{filename}_{quality}.mp4"
        out_path_drive = f"/content/drive/MyDrive/{filename}_{quality}.mp4"

        print(f"\n⬇ Downloading to Colab → {out_path_colab}")
        print("   (using ffmpeg to merge HLS segments...)\n")

        # 3. Download to Colab
        process = subprocess.Popen(
            ["ffmpeg", "-y", "-i", m3u8_url, "-c", "copy", "-bsf:a", "aac_adtstoasc", out_path_colab],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True
        )

        duration_secs = 0
        pbar = None

        for line in process.stdout:
            if not duration_secs:
                dur_match = re.search(r'Duration:\s*(\d+):(\d+):(\d+\.\d+)', line)
                if dur_match:
                    duration_secs = int(dur_match.group(1)) * 3600 + int(dur_match.group(2)) * 60 + float(dur_match.group(3))
                    pbar = tqdm(total=duration_secs, desc="Downloading", unit="s", bar_format="{l_bar}{bar}| {percentage:3.0f}%")

            if duration_secs and pbar:
                time_match = re.search(r'time=(\d+):(\d+):(\d+\.\d+)', line)
                if time_match:
                    time_secs = int(time_match.group(1)) * 3600 + int(time_match.group(2)) * 60 + float(time_match.group(3))
                    pbar.n = time_secs
                    pbar.refresh()

        if pbar:
            pbar.n = duration_secs
            pbar.refresh()
            pbar.close()

        process.wait()

        if process.returncode == 0:
            size_mb = os.path.getsize(out_path_colab) / 1024 / 1024
            print(f"\n✅ Done downloading to Colab! ({size_mb:.1f} MB)")

            # 4. Copy to Google Drive
            print(f"\n🔄 Copying to Google Drive → {out_path_drive}")
            shutil.copy(out_path_colab, out_path_drive)
            print("✅ Done copying to Google Drive!")

            # 5. Preview from Colab
            if input("\nPreview in notebook? (y/n): ").strip().lower() == "y":
                print(f"▶ Playing from: {out_path_colab}")
                display(Video(out_path_colab, embed=True, width=700))
        else:
            print(f"\n❌ ffmpeg failed (code {process.returncode})")

else:
    print("❌ Invalid — enter 1 or 2.")

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
🌐 Step 1: Fresh homepage visit (getting new session_id for this IP)...
   Status: 200
   New session cookies: ['session_id', '__secure_token']
   ⚠️  No cookie file — running as guest (5 req / 6h limit)

🔗 Paste your TeraBox / 1024terabox URL: https://1024terabox.com/s/12D7QorpJxHk9nuQgIDPeuA

Fetching video info...
   API status: 200  (attempt 1/3)
   📊 Usage: 3/5 (resets ~6h) — type: guest
   ⚠️  Still logged in as guest — login_token may be expired

📄 Name     : Money Heist S04E03 @filmysaga.mkv
📦 Size     : 219.61 MB
🎞  Quality  : 720p
⏱  Duration : 42:43
📡 Streams  : 360p, 480p, 720p

What do you want to do?
  1 → Watch  (show url to play video)
  2 → Download  (download via m3u8 stream to Colab & Drive)
Enter 1 or 2: 1

Available qualities: 360p, 480p, 720p
Pick quality (de